In [ ]:
# 1. Setup: get the code, install what the stages need, compile the viewer.
# Needs a T4 GPU runtime. Run the cells in order.
import glob, os, shutil, signal, subprocess, sys, time, urllib.request
from IPython.display import Image, display

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]

SURVEY = "CA_SantaClaraCounty_2020"       # USGS 3DEP survey name in the public lidar bucket
PLACE = dict(lat=37.4275, lon=-122.1700)   # Stanford Main Quad (the walk in make_route.py is drawn for this spot)
SIZE_M = 400       # side of the square, in metres
DEPTH = 12         # lidar detail

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
exp = os.path.join(repo, "experiments", "realplace")
build = os.path.join(repo, "build_pipeline")
lidar = os.path.join(build, "quad.bin")              # stage 1: what the laser measured
walls = os.path.join(build, "quad_walls.bin")        # stage 2: plus straight walls
route = os.path.join(build, "route.txt")             # stage 3: where the cameras stand
painted = os.path.join(build, "quad_painted.bin")    # stage 4: walls painted along the route


def sh(*cmd, cwd=None):
    """Run a command, print its output line by line as it comes, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)
sh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

sh(sys.executable, "-m", "pip", "install", "-q", "laspy[lazrs]", "diffusers", "transformers", "accelerate")
os.makedirs(build, exist_ok=True)
sh("nvcc", *FLAGS, "-I", os.path.join(repo, "src"), "-I", exp,
   os.path.join(exp, "viewer_real.cu"), "-o", os.path.join(build, "viewer_real"))
print("setup done")


In [ ]:
# 2. Stages 1 to 3: fetch the lidar, add the walls, lay out the route. About a minute, no GPU needed.
sh(sys.executable, os.path.join(exp, "fetch_place.py"), "--survey", SURVEY,
   "--lat", str(PLACE["lat"]), "--lon", str(PLACE["lon"]), "--size", str(SIZE_M), "--depth", str(DEPTH),
   "--view-distance", "260", "--view-height", "150", "--out", lidar)
sh(sys.executable, os.path.join(exp, "make_walls.py"), "--scene", lidar, "--out", walls)
sh(sys.executable, os.path.join(exp, "make_route.py"), "--scene", walls, "--out", route)
display(Image(route.replace(".txt", ".png")))    # blue = the walk, pink = the flyover


In [ ]:
# 3. Stage 4: paint the walls along the route with the image model.
# QUICK_TRY paints about 10 views spread along the route, to check the model works (a minute or two,
# plus a few minutes the first time to download the model). Set it to 0 to paint every view.
QUICK_TRY = 10

log = os.path.join(build, "paint_log")
shutil.rmtree(log, ignore_errors=True)
sh(sys.executable, os.path.join(exp, "paint_route.py"), "--scene", walls, "--route", route,
   "--out", painted, "--log", log, "--limit", str(QUICK_TRY))

# Each strip is: what the camera saw, the boxes that were painted (white), and the result.
strips = sorted(glob.glob(os.path.join(log, "*.jpg")))
for path in strips[::max(1, len(strips) // 6)][:6]:
    print(os.path.basename(path))
    display(Image(path))


In [ ]:
# 4. Fly through it. SHOW picks which scene:
#   "painted"     the result          "walls"  plain walls, before painting
#   "confidence"  red = never painted, yellow = painted from far or at a slant, green = painted well
#   "lidar"       only what the laser measured
SHOW = "painted"
PORT = 8002
DOT_SIZE_M = 0.3
FLY_SPEED = 12.0   # metres per second (Shift is 3 times faster)

scene = {"painted": painted, "walls": walls, "lidar": lidar,
         "confidence": painted.replace(".bin", "_confidence.bin")}[SHOW]
pid_file, log_file = "/content/pointwalk_pipeline.pid", "/content/pointwalk_pipeline.log"

if os.path.exists(pid_file):                      # stop the server from the last run of this cell
    try:
        os.killpg(int(open(pid_file).read()), signal.SIGTERM)
        time.sleep(1)
    except (ProcessLookupError, ValueError, PermissionError):
        pass
    os.remove(pid_file)

# The page and the server are the project's own. Only the fly speed is changed, in a copy.
page = open(os.path.join(repo, "viewer", "index.html")).read()
open(os.path.join(build, "index.html"), "w").write(
    page.replace("const WALK_SPEED = 3.0;", f"const WALK_SPEED = {FLY_SPEED};"))
shutil.copy(os.path.join(repo, "viewer", "server.py"), os.path.join(build, "server.py"))
wrapper = os.path.join(build, "run_viewer.sh")
open(wrapper, "w").write(f'#!/bin/sh\nexec "{os.path.join(build, "viewer_real")}" "$1" {DOT_SIZE_M}\n')
os.chmod(wrapper, 0o755)

server = subprocess.Popen(
    [sys.executable, os.path.join(build, "server.py"), "--renderer", wrapper, "--scene", scene,
     "--port", str(PORT), "--host", "0.0.0.0"],
    stdout=open(log_file, "w"), stderr=subprocess.STDOUT, start_new_session=True)
open(pid_file, "w").write(str(server.pid))

ready = False
for _ in range(240):
    if server.poll() is not None:
        break
    try:
        ready = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/ping", timeout=1).read() == b"ok"
        if ready:
            break
    except OSError:
        time.sleep(0.5)
print(open(log_file).read())
if not ready:
    raise SystemExit("The viewer server did not start. The lines above are its log.")

reply = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/frame?reset=1", timeout=30)
first = reply.read()
print("first picture:", "kernels", reply.headers.get("X-Kernel-Ms"), "ms,",
      "whole picture on GPU", reply.headers.get("X-Frame-Ms"), "ms,", reply.headers.get("X-Dots"), "dots")
display(Image(first))

from google.colab import output
try:
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the viewer in its own tab (use this if the box below stays empty)")
except Exception as problem:
    print("could not make the link:", problem)
try:
    output.serve_kernel_port_as_iframe(PORT, height="640")
except Exception as problem:
    print("could not show the page here:", problem)
